# Session 15 · Final Project and Next Steps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-15-final-project-and-next-steps.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

**Setup:** the Spark examples need PySpark (and Java, which Google Colab already has).

In [1]:
%pip install -q pyspark

### Example 1 · Reference pipeline: bronze → silver → gold

In [2]:
import os, tempfile
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[2]").appName("final-project")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "Asia/Karachi")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.session.timeZone", "Asia/Karachi")
PARAMS = {"base": tempfile.mkdtemp(), "start": "2026-09-01", "days": 30, "fare_per_km": 55.0}

# ---------- BRONZE: ingest raw data as-is ----------
zones = F.array(*[F.lit(z) for z in ["Clifton", "Saddar", "Gulshan", "DHA", "Korangi", "North Nazimabad"]])
raw = (spark.range(120000)
    .withColumn("pickup_ts", (F.unix_timestamp(F.lit(PARAMS["start"]).cast("timestamp"))
                              + (F.rand(seed=1) * PARAMS["days"] * 86400)).cast("timestamp"))
    .withColumn("zone", F.element_at(zones, (F.abs(F.hash("id")) % 6 + 1).cast("int")))
    .withColumn("km", F.round(F.exp(F.randn(seed=2) * 0.5 + 1.8), 1))
    .withColumn("fare", F.round(F.col("km") * PARAMS["fare_per_km"] + 120 + F.randn(seed=3) * 30, 0))
    .withColumn("km", F.when(F.rand(seed=4) < 0.005, -1.0).otherwise(F.col("km"))))     # bad records
raw = raw.unionByName(raw.limit(300))                                                     # duplicates
bronze = os.path.join(PARAMS["base"], "bronze", "trips")
raw.write.mode("overwrite").parquet(bronze)

# ---------- SILVER: clean, validate, partition ----------
b = spark.read.parquet(bronze)
silver_df = (b.dropDuplicates(["id"]).filter((F.col("km") > 0) & (F.col("fare") > 0))
              .withColumn("trip_date", F.to_date("pickup_ts"))
              .withColumn("hour", F.hour("pickup_ts")))
silver = os.path.join(PARAMS["base"], "silver", "trips")
silver_df.write.mode("overwrite").partitionBy("trip_date").parquet(silver)
n_b, n_s = b.count(), spark.read.parquet(silver).count()
print(f"bronze rows {n_b:,} -> silver rows {n_s:,} (removed {n_b - n_s:,} duplicates and invalid trips)")
print("silver date partitions:", len([d for d in os.listdir(silver) if d.startswith("trip_date=")]))

# ---------- GOLD: KPI tables for the dashboard ----------
s = spark.read.parquet(silver)
gold_zone = (s.groupBy("zone").agg(F.count("*").alias("trips"), F.round(F.avg("km"), 2).alias("avg_km"),
                                   F.round(F.sum("fare") / 1e6, 2).alias("revenue_m"))
              .orderBy(F.desc("trips")))
gold_zone.write.mode("overwrite").parquet(os.path.join(PARAMS["base"], "gold", "zone_kpis"))
gold_zone.show()
s.groupBy("hour").count().orderBy(F.desc("count")).limit(3).show()

bronze rows 120,300 -> silver rows 119,366 (removed 934 duplicates and invalid trips)
silver date partitions: 30


+---------------+-----+------+---------+
|           zone|trips|avg_km|revenue_m|
+---------------+-----+------+---------+
|North Nazimabad|20100|  6.86|     10.0|
|         Saddar|19948|  6.82|     9.89|
|        Gulshan|19911|  6.85|     9.89|
|            DHA|19861|   6.8|     9.81|
|        Korangi|19837|  6.85|     9.85|
|        Clifton|19709|  6.87|     9.81|
+---------------+-----+------+---------+



+----+-----+
|hour|count|
+----+-----+
|  23| 5182|
|   8| 5109|
|   4| 5058|
+----+-----+



### Example 2 · Model and data-quality checks on the silver table

In [3]:
import os, tempfile
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, OneHotEncoder, StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
spark = (SparkSession.builder.master("local[2]").appName("final-model")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

zones = F.array(*[F.lit(z) for z in ["Clifton", "Saddar", "Gulshan", "DHA", "Korangi", "North Nazimabad"]])
trips = (spark.range(60000)
    .withColumn("zone", F.element_at(zones, (F.abs(F.hash("id")) % 6 + 1).cast("int")))
    .withColumn("hour", (F.abs(F.hash(F.col("id") * 3)) % 24).cast("int"))
    .withColumn("km", F.round(F.exp(F.randn(seed=2) * 0.5 + 1.8), 1))
    .withColumn("fare", F.round(F.col("km") * 55 + 120 + F.when(F.col("hour").between(17, 20), 80).otherwise(0)
                                + F.randn(seed=3) * 30, 0)))

checks = trips.agg(F.sum(F.col("km").isNull().cast("int")).alias("null_km"),
                   F.sum((F.col("fare") <= 0).cast("int")).alias("non_positive_fares"),
                   F.min("km").alias("min_km"), F.max("km").alias("max_km"))
print("data-quality checks:", checks.first().asDict())

train, test = trips.randomSplit([0.8, 0.2], seed=42)       # not a time series, so a random split is fine
pipe = Pipeline(stages=[
    StringIndexer(inputCol="zone", outputCol="zone_idx"),
    OneHotEncoder(inputCols=["zone_idx"], outputCols=["zone_vec"]),
    VectorAssembler(inputCols=["km", "zone_vec"], outputCol="features"),
    LinearRegression(labelCol="fare")])
peak = F.col("hour").between(17, 20).cast("int")
model_plain = pipe.fit(train)
pipe_peak = Pipeline(stages=pipe.getStages()[:2] + [
    VectorAssembler(inputCols=["km", "zone_vec", "peak"], outputCol="features"), LinearRegression(labelCol="fare")])
model_peak = pipe_peak.fit(train.withColumn("peak", peak))

ev = RegressionEvaluator(labelCol="fare", metricName="rmse")
mean_fare = train.agg(F.avg("fare")).first()[0]
print(f"baseline RMSE (always predict mean fare): {ev.evaluate(test.withColumn('prediction', F.lit(mean_fare))):.1f}")
print(f"model RMSE (km + zone)                  : {ev.evaluate(model_plain.transform(test)):.1f}")
print(f"model RMSE (km + zone + peak hour)      : {ev.evaluate(model_peak.transform(test.withColumn('peak', peak))):.1f}")
lr = model_peak.stages[-1]
print(f"fare ~ {lr.intercept:.0f} + {lr.coefficients[0]:.1f} x km ... + {lr.coefficients[-1]:.0f} at peak hours")

data-quality checks: {'null_km': 0, 'non_positive_fares': 0, 'min_km': 0.6, 'max_km': 47.5}


baseline RMSE (always predict mean fare): 204.4


model RMSE (km + zone)                  : 42.6


model RMSE (km + zone + peak hour)      : 29.9
fare ~ 119 + 55.1 x km ... + 80 at peak hours
